# Bank Observability — the partial-observability appendix (task 5)

Reviewers R1-1 / R1-2: **no single institution observes the graph GARG-AML scores.** A
bank sees only the transactions with one of its own customers on them, so the
second-order neighbourhood the score is computed from may not exist from its point of
view.

This notebook produces the tables and figures for the appendix. It does **not** compute
anything expensive — `scripts/partial_observability.py` does that and writes two files
per institution, which this notebook reads:

| file | contents |
| --- | --- |
| `results/<view>_partial_observability_accounts.csv` | one row per (client, direction): the score under both regimes, degree, N₁/N₂ under both regimes, label propensities |
| `results/<view>_partial_observability_metrics.csv` | tidy metrics per (direction, regime, cut-off, target, subgroup, metric, K) |

**Run first:**

```bash
python scripts/partial_observability.py
```

### The comparison

For every client of the institution the score is computed twice: on the **full** raw
transaction graph, and on the **view** — the raw graph of the visible transactions only.
**Neither side is Louvain-reduced.** The published pipeline severs inter-community edges,
but full and view would then be reduced by *different* partitions and the degradation
could not be attributed to missing edges rather than to a changed partition.

> **The `full` column here is therefore not the published Table 10–11 number.** It is a
> baseline computed for this comparison, and the appendix must say so.

In [ ]:
# Run from the notebook directory; chdir to the repo root so the relative
# paths used elsewhere in the codebase work.
import os
import sys

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
REPO_ROOT = os.getcwd()
if REPO_ROOT not in sys.path:
    sys.path.append(REPO_ROOT)
print("cwd:", REPO_ROOT)

In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data.bank_views import (BANK_COLUMNS, ACCOUNT_COLUMNS, BANK_DTYPES,
                                 bank_account_pairs, bank_clients, client_counts,
                                 filter_transactions, resolve_banks, view_name)

DATASET = "HI-Small"
TRANS = f"data/{DATASET}_Trans.csv"

# Must match scripts/partial_observability.py.
INSTITUTIONS = ["012", "top50"]
DIRECTION = "undirected"      # the paper's headline score
MIN_DEGREE = 3
FIGDIR = Path("results")

plt.rcParams.update({"figure.dpi": 120, "font.size": 9})

## 1. Bank sizes on HI-Small

Establishes which bank is largest, and the two notions of "size" that disagree.

In [ ]:
pairs = bank_account_pairs(TRANS)
clients_per_bank = pairs.groupby("bank")["account"].nunique().sort_values(ascending=False)
volume = pd.concat([pd.read_csv(TRANS, usecols=[c], dtype=BANK_DTYPES)[c]
                    for c in BANK_COLUMNS], ignore_index=True).value_counts()

banks = pd.DataFrame({"clients": clients_per_bank, "transactions": volume}).fillna(0).astype(int)
N_ACC = pairs["account"].nunique()
N_TX = int(volume.sum() / 2)

print(f"HI-Small: {N_ACC:,} accounts, {N_TX:,} transactions, {len(banks):,} banks")
print(f"client counts: min {banks['clients'].min()}, median {banks['clients'].median():.0f}, "
      f"max {banks['clients'].max():,}")
print(f"banks with a single client: {(banks['clients'] == 1).sum():,}")
display(banks.sort_values("clients", ascending=False).head(10))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.4))

axes[0].hist(banks["clients"], bins=np.logspace(0, np.log10(banks["clients"].max()), 40),
             color="0.4")
axes[0].set_xscale("log"); axes[0].set_yscale("log")
axes[0].set_xlabel("clients per bank"); axes[0].set_ylabel("number of banks")
axes[0].set_title("(a) Bank size distribution")

axes[1].scatter(banks["clients"], banks["transactions"], s=5, alpha=0.25, color="0.4")
axes[1].set_xscale("log"); axes[1].set_yscale("log")
axes[1].set_xlabel("clients"); axes[1].set_ylabel("transactions")
axes[1].set_title("(b) Clients vs. volume")
biggest = banks.sort_values("transactions", ascending=False).index[0]
axes[1].annotate(f"bank {biggest}", (banks.loc[biggest, "clients"], banks.loc[biggest, "transactions"]),
                 textcoords="offset points", xytext=(6, -2), fontsize=8)

plt.tight_layout()
plt.savefig(FIGDIR / "appendix_bank_sizes.pdf", bbox_inches="tight")
plt.show()

**For the text.** The distribution is extremely fragmented, which is itself a
limitation worth stating: the *largest* bank in HI-Small holds a fraction of a percent of
all accounts, where a real large bank would hold a double-digit share. That is why the
appendix also reports the 50 largest banks pooled into one institution.

## 2. Coverage — the "x% of clients, y% of transactions" table

Table A.1 of the appendix.

In [ ]:
rows = []
for spec in INSTITUTIONS:
    banks_resolved = resolve_banks(spec, TRANS)
    visible = filter_transactions(
        pd.read_csv(TRANS, usecols=list(BANK_COLUMNS + ACCOUNT_COLUMNS), dtype=BANK_DTYPES),
        banks_resolved)
    cl = bank_clients(visible, banks_resolved)
    nodes = set(visible[ACCOUNT_COLUMNS[0]]) | set(visible[ACCOUNT_COLUMNS[1]])
    rows.append(dict(
        institution=spec, banks=len(banks_resolved),
        clients=len(cl), pct_clients=100 * len(cl) / N_ACC,
        transactions=len(visible), pct_transactions=100 * len(visible) / N_TX,
        view_accounts=len(nodes), pct_view_accounts=100 * len(nodes) / N_ACC,
        external=len(nodes) - len(cl),
    ))

coverage = pd.DataFrame(rows).set_index("institution")
coverage.to_csv(FIGDIR / "appendix_coverage.csv")
display(coverage.round(3))

for spec, r in coverage.iterrows():
    print(f"{spec}: {r['clients']:,.0f} clients = {r['pct_clients']:.3f}% of all accounts; "
          f"{r['transactions']:,.0f} transactions = {r['pct_transactions']:.3f}% of all transactions")

## 3. Neighbourhood survival — the mechanism

The control and the effect, in one table:

* **N₁ survival must be exactly 1.0.** A bank sees every transaction of its own clients,
  so their direct counterparties are fully observable. If this is not 1.0 something is
  wrong (the only legitimate exceptions are accounts held at two banks).
* **N₂ survival is what partial observability destroys**, and N₂ is what the block
  densities are computed from.

In [ ]:
def load_accounts(spec, direction=DIRECTION):
    path = FIGDIR / f"{view_name(DATASET, spec)}_partial_observability_accounts.csv"
    if not path.exists():
        raise FileNotFoundError(f"{path} -- run scripts/partial_observability.py first")
    df = pd.read_csv(path, dtype={"account": str}).set_index("account")
    return df[df["direction"] == direction]

accounts = {spec: load_accounts(spec) for spec in INSTITUTIONS
            if (FIGDIR / f"{view_name(DATASET, spec)}_partial_observability_accounts.csv").exists()}
print("loaded:", list(accounts))

In [ ]:
rows = []
for spec, df in accounts.items():
    n1_surv = (df["n1_view"] / df["n1_full"].replace(0, np.nan))
    n2_surv = (df["n2_view"] / df["n2_full"].replace(0, np.nan))
    rows.append(dict(
        institution=spec, clients=len(df),
        n1_survival_mean=n1_surv.mean(), n1_survival_min=n1_surv.min(),
        n2_full_median=df["n2_full"].median(), n2_view_median=df["n2_view"].median(),
        n2_survival_mean=n2_surv.mean(), n2_survival_median=n2_surv.median(),
        clients_losing_all_n2=int((df["n2_view"] == 0).sum()),
    ))

survival = pd.DataFrame(rows).set_index("institution")
survival.to_csv(FIGDIR / "appendix_neighbourhood_survival.csv")
display(survival.round(4))
print("N1 survival must be 1.0 everywhere -- it is the control, not a result.")

In [ ]:
fig, axes = plt.subplots(1, len(accounts), figsize=(4.2 * len(accounts), 3.4), squeeze=False)
for ax, (spec, df) in zip(axes[0], accounts.items()):
    surv = (df["n2_view"] / df["n2_full"].replace(0, np.nan)).dropna()
    ax.hist(surv, bins=np.linspace(0, 1, 26), color="0.4")
    ax.axvline(surv.mean(), color="black", ls="--", lw=1,
               label=f"mean {surv.mean():.2f}")
    ax.set_xlabel("fraction of $N_2$ still visible")
    ax.set_ylabel("clients")
    ax.set_title(f"institution {spec}")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(FIGDIR / "appendix_n2_survival.pdf", bbox_inches="tight")
plt.show()

## 4. Score impact

How far the score moves when the bank loses its second-order view. Rank correlation is
the number to quote: the score is used as a **ranking**, so what matters is whether the
same clients stay at the top, not whether the values are equal.

In [ ]:
from scipy.stats import pearsonr, spearmanr

rows = []
for spec, df in accounts.items():
    for name, keep in [("all", pd.Series(True, index=df.index)),
                       (f"degree>={MIN_DEGREE}", df["degree"] >= MIN_DEGREE)]:
        f, v = df.loc[keep, "GARGAML_full"], df.loc[keep, "GARGAML_view"]
        ok = f.notna() & v.notna()
        f, v = f[ok], v[ok]
        rows.append(dict(
            institution=spec, subgroup=name, clients=len(f),
            mean_full=f.mean(), mean_view=v.mean(), mean_shift=(v - f).mean(),
            pearson=pearsonr(f, v)[0], spearman=spearmanr(f, v)[0],
            unchanged=int(np.isclose(f, v).sum()),
            pct_unchanged=100 * np.isclose(f, v).mean(),
        ))

shift = pd.DataFrame(rows).set_index(["institution", "subgroup"])
shift.to_csv(FIGDIR / "appendix_score_shift.csv")
display(shift.round(4))

In [ ]:
fig, axes = plt.subplots(2, len(accounts), figsize=(4.2 * len(accounts), 6.4), squeeze=False)

for col, (spec, df) in enumerate(accounts.items()):
    keep = df["degree"] >= MIN_DEGREE
    f, v = df.loc[keep, "GARGAML_full"], df.loc[keep, "GARGAML_view"]

    ax = axes[0][col]
    ax.scatter(f, v, s=6, alpha=0.2, color="0.3")
    lims = (-1.05, 1.05)
    ax.plot(lims, lims, ls="--", lw=1, color="black")
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_xlabel("score, full network"); ax.set_ylabel("score, bank view")
    ax.set_title(f"{spec}  (degree $\\geq$ {MIN_DEGREE}, $\\rho$={spearmanr(f, v)[0]:.2f})")

    ax = axes[1][col]
    bins = np.linspace(-1, 1, 41)
    ax.hist(f, bins=bins, alpha=0.6, label="full network", color="0.25")
    ax.hist(v, bins=bins, alpha=0.6, label="bank view", color="0.65")
    ax.set_xlabel("GARG-AML score"); ax.set_ylabel("clients")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(FIGDIR / "appendix_score_shift.pdf", bbox_inches="tight")
plt.show()

## 5. Detection impact

Table A.2. Read `ties@K` beside every ranking metric: the score takes few distinct values
on a single institution's clients, so when the tie group at the cut-off is larger than
*K*, "the top *K*" is not uniquely defined and the metric at that *K* is not
interpretable on its own.

In [ ]:
def load_metrics(spec):
    path = FIGDIR / f"{view_name(DATASET, spec)}_partial_observability_metrics.csv"
    return pd.read_csv(path)

metrics = {spec: load_metrics(spec) for spec in accounts}

def detection_table(spec, direction=DIRECTION, subgroup="all", target="Is Laundering"):
    """full vs view, one row per (cut-off, metric)."""
    m = metrics[spec]
    m = m[(m["direction"] == direction) & (m["subgroup"] == subgroup)
          & (m["target"] == target) & (m["status"] == "ok")]
    m = m.assign(name=np.where(m["K"].notna(),
                               m["metric"].str.replace("@K", "", regex=False)
                               + "@" + m["K"].fillna(0).astype(int).astype(str),
                               m["metric"]))
    wide = m.pivot_table(index=["cutoff", "name"], columns="regime", values="value")
    wide["change"] = wide["view"] - wide["full"]
    return wide

for spec in accounts:
    print(f"\n===== institution {spec} -- {DIRECTION}, all clients =====")
    t = detection_table(spec)
    keep = [n for n in ["AUC_ROC", "AUC_PR", "P@50", "R@50", "lift@50", "TP@50",
                        "P@100", "lift@100", "ties@50", "ties@100"]
            if n in t.index.get_level_values("name")]
    display(t.loc[(slice(None), keep), :].round(4))

In [ ]:
# The same table on the non-degenerate subgroup, where top-K is better defined.
for spec in accounts:
    print(f"\n===== institution {spec} -- {DIRECTION}, degree >= {MIN_DEGREE} =====")
    t = detection_table(spec, subgroup=f"degree>={MIN_DEGREE}")
    keep = [n for n in ["AUC_ROC", "AUC_PR", "P@50", "R@50", "lift@50",
                        "ties@50", "ties@100"]
            if n in t.index.get_level_values("name")]
    display(t.loc[(slice(None), keep), :].round(4))

In [ ]:
# Headline figure: AUC-ROC and AUC-PR, full vs view, per institution and cut-off.
head = []
for spec, m in metrics.items():
    sel = m[(m["direction"] == DIRECTION) & (m["subgroup"] == "all")
            & (m["target"] == "Is Laundering") & (m["status"] == "ok")
            & (m["metric"].isin(["AUC_ROC", "AUC_PR"]))]
    for _, r in sel.iterrows():
        head.append(dict(institution=spec, cutoff=r["cutoff"], metric=r["metric"],
                         regime=r["regime"], value=r["value"]))
head = pd.DataFrame(head)

if not head.empty:
    cells = head.groupby(["metric", "institution", "cutoff"])
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.4))
    for ax, metric in zip(axes, ["AUC_ROC", "AUC_PR"]):
        sub = head[head["metric"] == metric]
        labels = sorted({(i, c) for i, c in zip(sub["institution"], sub["cutoff"])})
        x = np.arange(len(labels))
        for offset, regime, colour in [(-0.18, "full", "0.25"), (0.18, "view", "0.65")]:
            vals = [sub[(sub["institution"] == i) & (sub["cutoff"] == c)
                        & (sub["regime"] == regime)]["value"].mean() for i, c in labels]
            ax.bar(x + offset, vals, width=0.34, label=regime, color=colour)
        ax.set_xticks(x)
        ax.set_xticklabels([f"{i}\n@{c:g}" for i, c in labels], fontsize=8)
        ax.set_ylabel(metric.replace("_", "-"))
        ax.axhline(0.5 if metric == "AUC_ROC" else 0, color="black", lw=0.8, ls=":")
        ax.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(FIGDIR / "appendix_detection.pdf", bbox_inches="tight")
    plt.show()

## 6. What goes in the appendix

Everything above writes to `results/`:

| output | appendix element |
| --- | --- |
| `appendix_coverage.csv` | Table A.1 — how much of the network one institution sees |
| `appendix_neighbourhood_survival.csv` | Table A.2 — N₁ (control) and N₂ survival |
| `appendix_score_shift.csv` | Table A.3 — mean shift, Pearson/Spearman |
| `appendix_bank_sizes.pdf` | Fig. A.x — bank size distribution, clients vs. volume |
| `appendix_n2_survival.pdf` | Fig. A.x — distribution of N₂ survival |
| `appendix_score_shift.pdf` | Fig. A.x — full-vs-view scatter and score distributions |
| `appendix_detection.pdf` | Fig. A.x — AUC-ROC / AUC-PR, full vs view |

**Three things the text must state.**

1. The `full` baseline here is computed **without** Louvain and is therefore not the
   published Tables 10–11 number.
2. N₁ survival is 1.0 **by construction**, not by measurement — it is the control that
   confirms the view is built correctly, and it is why the comparison is paired
   (same clients, same labels, same degree subgroup on both sides).
3. AMLworld's banks are small: the largest holds well under 1% of accounts. The pooled
   `top50` institution exists to give a realistic market share and enough positives, and
   the single-bank result should be read as a lower bound on institution size, not as a
   typical bank.